# 12 — E3C Analysis | Semantic Weight Sensitivity (SNA-MXH / Group 14)

**Read-only inputs, no training, no checkpoint changes.** Follow-up to `11_E3C_Semantic_Weight_Sensitivity.ipynb`, `E3C_Export_TopK_Fixed.ipynb`, `09_Audit.ipynb`. Uses 5 weights × 3 seeds, train-only masked Top-20 exports, E2 head = top 20% by train degree (ties item ID), conditional macro head/tail recall, sample SD (`ddof=1`).

**Execution:** Google Colab → run cells top to bottom. Stop on any QA failure. Results are written only to a **new** `experiments/E3C/analysis/<timestamp>` folder. Do not treat results as verified until QA and baseline checks pass. The saved E2 protocol is the source for popularity definitions; interpretation does not claim statistical significance from 3 seeds.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os, json, csv, pickle, hashlib, math
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

ROOT = Path('/content/drive/MyDrive/Data_SNA_Group14')
DATA = ROOT / 'data/raw/amazon'
E3C = ROOT / 'experiments/E3C'
E1 = ROOT / 'experiments/predictions/lightgcn_plus'
WEIGHTS = ('0', '0.005', '0.01', '0.02', '0.05')
SEEDS = (2023, 2024, 2025)
KS = (10, 20)
N_USERS, N_ITEMS, N_TEST_USERS = 11000, 9332, 9860
# STRICT: abort on missing files, invalid masks, mismatched baseline.
STRICT_BASELINE = True
BASELINE_RECALL20_EXPECTED = 0.148863 # E3B_RQ2_summary.txt, rounded to 6 decimals
BASELINE_TOL = 2e-5
assert ROOT.is_dir() and DATA.is_dir() and E3C.is_dir(), 'Project root/data/E3C missing. Check Drive mount.'
print('Project:', ROOT)
print('Mode: ANALYSIS ONLY; no training or overwriting source files')

## 1. Load fixed protocol and train/test matrices
Only unpickle **trusted group-owned** `.pkl` files. The test matrix is used **only for evaluation**, never to rank items.

In [ ]:
protocol_path = E3C/'E3C_protocol.json'
assert protocol_path.is_file(), f'Missing protocol: {protocol_path}'
protocol = json.loads(protocol_path.read_text(encoding='utf-8'))
assert protocol['model'] == 'lightgcn_plus' and protocol['dataset'] == 'amazon'
assert [float(w) for w in WEIGHTS] == [float(w) for w in protocol['weights']]
assert list(SEEDS) == list(protocol['seeds'])
assert protocol['evaluation']['ranking_mask'] == 'train_only'
assert protocol['evaluation']['validation_masked'] is False
assert protocol['architecture']['graph_layers'] == 3
assert protocol['architecture']['embedding_dimension'] == 32
with open(DATA/'trn_mat.pkl','rb') as f: train = pickle.load(f)
with open(DATA/'tst_mat.pkl','rb') as f: test = pickle.load(f)
assert sp.issparse(train) and sp.issparse(test), 'Expected sparse train/test matrices'
train = train.tocsr(copy=True); test = test.tocsr(copy=True)
train.sum_duplicates(); test.sum_duplicates()
train.eliminate_zeros(); test.eliminate_zeros()
train.sort_indices(); test.sort_indices()
assert train.shape == test.shape == (N_USERS, N_ITEMS)
train.data[:] = 1; test.data[:] = 1
users_expected = np.flatnonzero(np.diff(test.indptr) > 0)
assert len(users_expected) == N_TEST_USERS
item_degree = np.asarray(train.getnnz(axis=0)).ravel()
head_count = math.ceil(.2*N_ITEMS)
# Degree descending; equal-degree ties by item ID ascending.
head_order = np.lexsort((np.arange(N_ITEMS), -item_degree))
head_mask = np.zeros(N_ITEMS, dtype=bool); head_mask[head_order[:head_count]] = True
assert head_mask.sum() == head_count
print('Dataset:', train.shape, '| test users:', len(users_expected), '| head items:', head_count)

## 2. Resolve 15 Top-K exports and validate provenance

Uses `E3C/runs/full/weight_*/seed_*/topk/topk_test.npz`; `weight=0.01` may fall back to original E1 export **only if** E3C copy is absent. For all other weights, missing exports stop execution. Checks shape, exact user order, IDs, unique recommendations, descending finite scores, train-mask exclusion, and available verification/provenance. A saved NPZ cannot by itself prove that ranking code never used test labels; source exporter was audited separately.

In [ ]:
def sha256(path, block=2**20):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(block),b''): h.update(chunk)
    return h.hexdigest()

def locate_topk(w,seed):
    e3c_path=E3C/'runs/full'/f'weight_{w}'/f'seed_{seed}'/'topk/topk_test.npz'
    if e3c_path.is_file(): return e3c_path, 'E3C'
    if w=='0.01':
        e1_path=E1/f'seed_{seed}'/'topk_test.npz'
        if e1_path.is_file(): return e1_path, 'E1_original_reuse'
    raise FileNotFoundError(f'Missing Top-K: weight={w}, seed={seed}: {e3c_path}')

def check_topk(path, weight, seed, origin):
    with np.load(path,allow_pickle=False) as z:
        assert {'user_ids','item_ids','scores'} <= set(z.files), (path,z.files)
        u=z['user_ids']; items=z['item_ids']; scores=z['scores']
    assert u.dtype.kind in 'iu' and items.dtype.kind in 'iu'
    assert u.shape == (N_TEST_USERS,) and items.shape == scores.shape == (N_TEST_USERS,20)
    assert np.array_equal(u, users_expected), f'Wrong test user IDs/order: {path}'
    assert np.all((items>=0)&(items<N_ITEMS)), f'Invalid item IDs: {path}'
    assert np.all(np.isfinite(scores)) and np.all(scores[:,:-1] >= scores[:,1:]-1e-6), f'Bad scores: {path}'
    assert np.all(np.diff(np.sort(items,axis=1),axis=1)!=0), f'Duplicate items: {path}'
    # Efficient sparse membership for all ~197k recommendations.
    flat_u=np.repeat(u,20); flat_i=items.ravel()
    assert not np.asarray(train[flat_u,flat_i]).any(), f'Train item leaked into recommendations: {path}'
    meta_dir=path.parent
    for name in ('export_verification.json','E3C_export_provenance.json'):
        p=meta_dir/name
        if p.is_file():
            d=json.loads(p.read_text(encoding='utf-8'))
            assert d.get('status') == 'passed', f'{p}: not passed'
            if name=='E3C_export_provenance.json':
                assert float(d.get('weight'))==float(weight) and int(d.get('seed'))==seed, f'Provenance mismatch: {p}'
                assert d.get('ranking_mask')=='train_only' and d.get('validation_masked') is False
    if origin=='E3C':
        assert (meta_dir/'E3C_export_provenance.json').is_file(), f'Missing E3C provenance: {path}'
    return u, items, {'weight':float(weight),'seed':seed,'origin':origin,'path':str(path),
                      'sha256':sha256(path),'qa':'PASS'}

exports={}; inventory=[]
for w in WEIGHTS:
    for seed in SEEDS:
        path,origin=locate_topk(w,seed)
        u,items,rec=check_topk(path,w,seed,origin)
        exports[(w,seed)]=items
        inventory.append(rec)
inventory_df=pd.DataFrame(inventory)
assert len(inventory_df)==15 and inventory_df.qa.eq('PASS').all()
print(inventory_df[['weight','seed','origin','qa']].to_string(index=False))

## 3. Compute accuracy and E2 popularity metrics

**Recall/NDCG:** macro average across 9,860 test users, binary test interactions. **ARP:** mean train item degree among recommended items. **APLT:** fraction of recommendations in tail 80%. **Head/Tail Recall:** conditional macro recall over users with at least one test item in the respective group. **Coverage:** distinct recommended items / 9,332. **Degree–exposure Spearman:** rank correlation between item train degree and Top-K exposure count across all items. All metrics computed separately at K=10 and K=20.

In [ ]:
def evaluate(items,k):
    rec=items[:,:k]
    u=users_expected
    truth=test[u]
    truth_count=np.asarray(truth.getnnz(axis=1)).ravel()
    # Pairwise sparse lookup: CSR indexed values, one boolean for each recommendation.
    hits=np.asarray(test[np.repeat(u,k),rec.ravel()]).reshape(len(u),k).astype(bool)
    hit_count=hits.sum(axis=1)
    recall=float(np.mean(hit_count/truth_count))
    discounts=1/np.log2(np.arange(2,k+2))
    dcg=hits @ discounts
    idcg=np.cumsum(discounts)[np.minimum(truth_count,k)-1]
    ndcg=float(np.mean(dcg/idcg))
    head_truth=np.asarray(truth[:,np.flatnonzero(head_mask)].getnnz(axis=1)).ravel()
    tail_truth=truth_count-head_truth
    head_hits=(hits & head_mask[rec]).sum(axis=1)
    tail_hits=(hits & ~head_mask[rec]).sum(axis=1)
    assert (head_hits+tail_hits==hit_count).all()
    head_valid=head_truth>0; tail_valid=tail_truth>0
    exposure=np.bincount(rec.ravel(),minlength=N_ITEMS)
    rho=spearmanr(item_degree,exposure).statistic
    return dict(K=k, Recall=recall, NDCG=ndcg,
                ARP=float(item_degree[rec].mean()),
                APLT=float((~head_mask[rec]).mean()),
                HeadRecall=float(np.mean(head_hits[head_valid]/head_truth[head_valid])),
                TailRecall=float(np.mean(tail_hits[tail_valid]/tail_truth[tail_valid])),
                Coverage=float(np.count_nonzero(exposure)/N_ITEMS),
                DegreeExposureSpearman=float(rho),
                n_head_eligible=int(head_valid.sum()),n_tail_eligible=int(tail_valid.sum()))
rows=[]
for (w,seed),items in exports.items():
    for k in KS:
        rows.append(dict(weight=float(w),seed=seed,**evaluate(items,k)))
run_df=pd.DataFrame(rows).sort_values(['weight','seed','K']).reset_index(drop=True)
assert len(run_df)==30 and run_df[['Recall','NDCG','ARP','APLT','HeadRecall','TailRecall','Coverage']].notna().all().all()
print(run_df[['weight','seed','K','Recall','NDCG','ARP','APLT','TailRecall']].to_string(index=False))

## 4. Baseline cross-check (weight=0.01 reused E1)
Checks against the saved E3B RQ2 summary. If this fails, **do not interpret the weight sweep**; inspect metric definitions and source data.

In [ ]:
baseline=run_df.query('weight == 0.01 and K == 20')
assert len(baseline)==3
baseline_mean=float(baseline.Recall.mean())
print('Recomputed E1/E3B Recall@20:',baseline_mean,'| E3B reported:',BASELINE_RECALL20_EXPECTED)
if STRICT_BASELINE:
    assert abs(baseline_mean-BASELINE_RECALL20_EXPECTED) <= BASELINE_TOL,         'E3B baseline mismatch: STOP; review definitions/exports before interpreting.'
print('Baseline cross-check:', 'PASS' if abs(baseline_mean-BASELINE_RECALL20_EXPECTED)<=BASELINE_TOL else 'REVIEW')

## 5. Aggregate mean ± sample SD; paired seed differences vs λ=0.01
Paired comparisons are descriptive, **not significance tests**.

In [ ]:
METRICS=['Recall','NDCG','ARP','APLT','HeadRecall','TailRecall','Coverage','DegreeExposureSpearman']
summary=run_df.groupby(['weight','K'],as_index=False).agg(**{
    f'{m}_{stat}':pd.NamedAgg(column=m,aggfunc=func)
    for m in METRICS for stat,func in [('mean','mean'),('sd',lambda x:x.std(ddof=1))]
})
summary['n_seeds']=3
baseline_by_seed=run_df[run_df.weight==0.01].set_index(['seed','K'])
paired=run_df.copy()
for m in METRICS:
    paired[f'delta_{m}']=paired.apply(lambda r:r[m]-baseline_by_seed.loc[(int(r.seed),int(r.K)),m],axis=1)
paired_summary=paired.groupby(['weight','K'],as_index=False)[[f'delta_{m}' for m in METRICS]].agg(['mean','std'])
paired_summary.columns=['_'.join(str(x) for x in col if x!='') for col in paired_summary.columns.to_flat_index()]
print(summary[['weight','K','Recall_mean','Recall_sd','NDCG_mean','ARP_mean','APLT_mean','TailRecall_mean']].to_string(index=False))

## 6. Save new analysis outputs and plots
Writes a new timestamped directory only after all QA and baseline checks. Never modifies E1/E2/E3B or E3C training/export files.

In [ ]:
stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
out=E3C/'analysis'/stamp
out.mkdir(parents=True,exist_ok=False)
inventory_df.to_csv(out/'E3C_topk_inventory_QA.csv',index=False)
run_df.to_csv(out/'E3C_seed_metrics.csv',index=False,float_format='%.10g')
summary.to_csv(out/'E3C_weight_summary.csv',index=False,float_format='%.10g')
paired.to_csv(out/'E3C_paired_seed_deltas.csv',index=False,float_format='%.10g')
paired_summary.to_csv(out/'E3C_paired_delta_summary.csv',index=False,float_format='%.10g')

for k in KS:
    d=summary[summary.K==k].sort_values('weight')
    fig,ax=plt.subplots(figsize=(8,5))
    ax.errorbar(d.weight,d.Recall_mean,yerr=d.Recall_sd,marker='o',capsize=4,label=f'Recall@{k}')
    ax.errorbar(d.weight,d.NDCG_mean,yerr=d.NDCG_sd,marker='s',capsize=4,label=f'NDCG@{k}')
    ax.set(xlabel='Semantic alignment weight (λ)',ylabel='Mean metric ± sample SD',title=f'E3C Accuracy vs Weight (K={k})')
    ax.grid(alpha=.25); ax.legend(); fig.tight_layout()
    fig.savefig(out/f'E3C_accuracy_weight_K{k}.png',dpi=180); plt.close(fig)
    fig,ax=plt.subplots(figsize=(8,5))
    ax.errorbar(d.weight,d.APLT_mean,yerr=d.APLT_sd,marker='o',capsize=4,label=f'APLT@{k}')
    ax.errorbar(d.weight,d.TailRecall_mean,yerr=d.TailRecall_sd,marker='s',capsize=4,label=f'Tail Recall@{k}')
    ax.set(xlabel='Semantic alignment weight (λ)',ylabel='Mean metric ± sample SD',title=f'E3C Tail Exposure / Recall (K={k})')
    ax.grid(alpha=.25); ax.legend(); fig.tight_layout()
    fig.savefig(out/f'E3C_tail_weight_K{k}.png',dpi=180); plt.close(fig)

report={'experiment':'E3C_Semantic_Weight_Sensitivity','status':'ANALYSIS_QA_PASS',
        'created_utc':datetime.now(timezone.utc).isoformat(),'protocol_source':str(protocol_path),
        'source_commit':protocol['source_commit'],'weights':[float(w) for w in WEIGHTS],
        'seeds':list(SEEDS),'n_exports':len(inventory_df),'n_metric_rows':len(run_df),
        'baseline_recall20_recomputed':baseline_mean,
        'baseline_recall20_reference':BASELINE_RECALL20_EXPECTED,
        'baseline_tolerance':BASELINE_TOL,'head_count':head_count,
        'head_rule':'top ceil(20%) by train degree; ties item ID ascending',
        'mask_rule':'train-only; source exporter provenance required for E3C',
        'limitations':['Top-K QA does not independently prove no test-label leakage in source ranking code',
                       'Three seeds do not establish statistical significance',
                       'Conditional macro head/tail recall excludes users without group test positives']}
(out/'E3C_analysis_protocol_and_QA.json').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
print('ANALYSIS_QA_PASS | Results:',out)
print('Saved files:',*[p.name for p in sorted(out.iterdir())],sep='\n - ')

## 7. Interpretation checklist (manual)

1. Confirm QA pass and baseline agreement. If a check fails, **do not use** the outputs in the report.
2. For each K, discuss Recall/NDCG together with ARP/APLT and conditional Head/Tail Recall. A lower ARP and higher APLT generally reflect greater tail exposure, but may trade off against accuracy.
3. Report mean ± sample SD across three seeds and paired seed deltas relative to λ=0.01. Do **not** infer causality or significance.
4. Inspect whether results are monotonic or non-monotonic; do not assume any weight is optimal before viewing computed tables.
5. This notebook does not run E4 and does not retrain E3C.